# E-Commerce Customer Analytics — Data Preparation & Cleaning

## Project Overview
This notebook establishes the foundational **Data Preparation & Cleansing Pipeline** for the **Brazilian E-Commerce Customer Analytics & Retention Intelligence** project using the Olist Public Dataset.

### Core Objectives:
1. **Systematic Ingestion**: Robustly load and audit all 9 relational tables.
2. **Quality Audit**: Identify missing values, duplicate entries, data types, and structural inconsistencies.
3. **Data Cleansing**: Implement documented, business-aligned cleaning decisions for orders, products, payments, reviews, and items.
4. **Relational Harmonization**: Construct an analytical master dataset (`df_master`) linking orders, customers, items, payments, and product hierarchies without grain mismatch or revenue duplication.
5. **Feature Engineering**: Compute operational KPIs (delivery duration, SLA compliance), financial indicators (order revenue), and temporal attributes (cohort periods).
6. **Customer Aggregation**: Build a unified customer behavioral profile (`df_customers`) capturing lifetime spend, purchase frequency, AOV, and repeat purchase classification.
7. **Production Export**: Export standardized datasets for exploratory data analysis, RFM customer segmentation, cohort retention, and BI reporting.

## Table of Contents
1. [1. Project Overview & Environment Setup](#1-project-overview--environment-setup)
2. [2. Configuration & Dataset Verification](#2-configuration--dataset-verification)
3. [3. Section 1: Data Loading & Initial Exploration](#3-section-1-data-loading--initial-exploration)
   - [3.1 Customers Dataset](#31-customers-dataset-olist_customers_datasetcsv)
   - [3.2 Orders Dataset](#32-orders-dataset-olist_orders_datasetcsv)
   - [3.3 Order Items Dataset](#33-order-items-dataset-olist_order_items_datasetcsv)
   - [3.4 Order Payments Dataset](#34-order-payments-dataset-olist_order_payments_datasetcsv)
   - [3.5 Order Reviews Dataset](#35-order-reviews-dataset-olist_order_reviews_datasetcsv)
   - [3.6 Products Dataset](#36-products-dataset-olist_products_datasetcsv)
   - [3.7 Sellers Dataset](#37-sellers-dataset-olist_sellers_datasetcsv)
   - [3.8 Geolocation Dataset](#38-geolocation-dataset-olist_geolocation_datasetcsv)
   - [3.9 Product Category Translation](#39-product-category-name-translation-product_category_name_translationcsv)
4. [4. Section 2: Data Quality Assessment](#4-section-2-data-quality-assessment)
   - [4.1 Unified Quality Matrix](#41-unified-data-quality-summary-matrix)
   - [4.2 Missing Value Deep Dive](#42-detailed-missing-value-breakdown)
   - [4.3 Numerical Distributions](#43-statistical-summaries-of-key-numeric-fields)
5. [5. Section 3: Data Cleaning Decisions & Implementation](#5-section-3-data-cleaning-decisions--implementation)
   - [5.1 Orders Cleaning](#51-cleaning-orders-olist_orders_dataset)
   - [5.2 Products Cleaning](#52-cleaning-products-olist_products_dataset)
   - [5.3 Reviews Cleaning](#53-cleaning-order-reviews-olist_order_reviews_dataset)
   - [5.4 Items Cleaning](#54-cleaning-order-items-olist_order_items_dataset)
   - [5.5 Payments Cleaning](#55-cleaning-order-payments-olist_order_payments_dataset)
6. [6. Section 4: Data Merging & Analytical Master Dataset Construction](#6-section-4-data-merging--analytical-master-dataset-construction)
7. [7. Section 5: Feature Engineering](#7-section-5-feature-engineering)
8. [8. Section 6: Customer-Level Aggregation](#8-section-6-customer-level-aggregation)
9. [9. Section 7: Export Clean Datasets](#9-section-7-export-clean-datasets)
10. [10. Section 8: Summary Table & Decision Audit Log](#10-section-8-summary-table--decision-audit-log)

## 1. Project Overview & Environment Setup
Import essential libraries for data manipulation, numerical processing, date/time operations, and tabular visualization.

In [ ]:
# Core scientific computing and data manipulation libraries
import os
import sys
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Pandas display formatting for comprehensive audits
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
warnings.filterwarnings('ignore')

print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")
print("✓ Environment successfully initialized.")

## 2. Configuration & Dataset Verification
Define directory paths (`BASE_DIR`, `DATA_DIR`, `EXPORTS_DIR`) relative to this notebook's location in `notebooks/`. Ensure export directories exist and audit file availability.

In [ ]:
# Setup paths relative to current notebook location
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
DATA_DIR = os.path.join(BASE_DIR, 'data')
EXPORTS_DIR = os.path.join(BASE_DIR, 'exports')

# Ensure exports directory exists
os.makedirs(EXPORTS_DIR, exist_ok=True)

print(f"Base Directory:    {BASE_DIR}")
print(f"Data Directory:    {DATA_DIR}")
print(f"Exports Directory: {EXPORTS_DIR}")

### Dataset Verification
Check for the presence and file size of all 9 expected CSV files in the `data/` folder before commencing data ingestion.

In [ ]:
REQUIRED_DATASETS = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'product_category_name_translation.csv'
]

verification_rows = []
all_files_found = True

for fname in REQUIRED_DATASETS:
    fpath = os.path.join(DATA_DIR, fname)
    exists = os.path.exists(fpath)
    size_mb = os.path.getsize(fpath) / (1024 * 1024) if exists else 0.0
    verification_rows.append({
        'Dataset': fname,
        'Status': '✅ Present' if exists else '❌ Missing',
        'Size (MB)': f"{size_mb:.2f}" if exists else "0.00"
    })
    if not exists:
        all_files_found = False

df_verification = pd.DataFrame(verification_rows)
display(df_verification)

if not all_files_found:
    print("\n⚠️ Note: Some CSV files are missing from data/.")
    print("Please follow data/README.md instructions to download the dataset from Kaggle.")
else:
    print("\n🎉 All 9 dataset files are present and verified!")

## Section 1: Data Loading & Initial Exploration
Load each of the 9 Olist CSV files with defensive exception handling. Inspect dataframe shapes, column data types, and initial records.

In [ ]:
def load_dataset(filename, parse_dates=None):
    """
    Safely load a CSV file from DATA_DIR.
    Returns an empty DataFrame if the file is not found or fails to load.
    """
    fpath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(fpath):
        print(f"⚠️ [NOT FOUND] {filename} does not exist at {fpath}")
        return pd.DataFrame()
    try:
        df = pd.read_csv(fpath, parse_dates=parse_dates)
        print(f"✓ Loaded {filename:<38} | {df.shape[0]:>8,} rows, {df.shape[1]:>2} columns")
        return df
    except Exception as e:
        print(f"❌ [ERROR] Could not load {filename}: {e}")
        return pd.DataFrame()

print("Beginning data ingestion...")
customers = load_dataset('olist_customers_dataset.csv')
orders = load_dataset('olist_orders_dataset.csv')
order_items = load_dataset('olist_order_items_dataset.csv')
order_payments = load_dataset('olist_order_payments_dataset.csv')
order_reviews = load_dataset('olist_order_reviews_dataset.csv')
products = load_dataset('olist_products_dataset.csv')
sellers = load_dataset('olist_sellers_dataset.csv')
geolocation = load_dataset('olist_geolocation_dataset.csv')
category_translation = load_dataset('product_category_name_translation.csv')
print("\nIngestion phase complete.")

### 3.1 Customers Dataset (`olist_customers_dataset.csv`)
- **Key Columns**:
  - `customer_id`: Unique identifier per order/transaction (foreign key referencing `orders.customer_id`).
  - `customer_unique_id`: Unique persistent identifier for each physical customer. Crucial for tracking repeat purchases across multiple orders, customer lifetime value (CLV), and retention analysis.
  - `customer_zip_code_prefix`, `customer_city`, `customer_state`: Demographic/geographic coordinates.

In [ ]:
print("--- Customers Dataset ---")
print(f"Shape: {customers.shape}")
print("\nData Types:")
print(customers.dtypes)
print("\nSample Records:")
display(customers.head(3))

### 3.2 Orders Dataset (`olist_orders_dataset.csv`)
- **Key Columns**:
  - `order_id`: Primary key of each customer order.
  - `customer_id`: Foreign key referencing `customers`.
  - `order_status`: Transaction status (`delivered`, `shipped`, `canceled`, `invoiced`, `processing`, `unavailable`, `approved`, `created`).
  - Timestamps: `order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`.

In [ ]:
print("--- Orders Dataset ---")
print(f"Shape: {orders.shape}")
print("\nData Types:")
print(orders.dtypes)
print("\nSample Records:")
display(orders.head(3))

### 3.3 Order Items Dataset (`olist_order_items_dataset.csv`)
- **Key Columns**:
  - `order_id`: Foreign key referencing `orders`.
  - `order_item_id`: Sequential item index within the order (1, 2, 3...).
  - `product_id`: Foreign key referencing `products`.
  - `seller_id`: Foreign key referencing `sellers`.
  - `price`: Item price in BRL.
  - `freight_value`: Shipping charge in BRL allocated to the item.

In [ ]:
print("--- Order Items Dataset ---")
print(f"Shape: {order_items.shape}")
print("\nData Types:")
print(order_items.dtypes)
print("\nSample Records:")
display(order_items.head(3))

### 3.4 Order Payments Dataset (`olist_order_payments_dataset.csv`)
- **Key Columns**:
  - `order_id`: Foreign key referencing `orders`.
  - `payment_sequential`: Sequence number of payment attempt/transaction.
  - `payment_type`: Method of payment (`credit_card`, `boleto`, `voucher`, `debit_card`).
  - `payment_installments`: Selected installment tenure.
  - `payment_value`: Monetary transaction value.

In [ ]:
print("--- Order Payments Dataset ---")
print(f"Shape: {order_payments.shape}")
print("\nData Types:")
print(order_payments.dtypes)
print("\nSample Records:")
display(order_payments.head(3))

### 3.5 Order Reviews Dataset (`olist_order_reviews_dataset.csv`)
- **Key Columns**:
  - `review_id`: Unique review survey ID.
  - `order_id`: Foreign key referencing `orders`.
  - `review_score`: Satisfaction rating from 1 (very dissatisfied) to 5 (very satisfied).
  - `review_comment_title`, `review_comment_message`: Unstructured customer comments.
  - `review_creation_date`, `review_answer_timestamp`: Review lifecycle timestamps.

In [ ]:
print("--- Order Reviews Dataset ---")
print(f"Shape: {order_reviews.shape}")
print("\nData Types:")
print(order_reviews.dtypes)
print("\nSample Records:")
display(order_reviews.head(3))

### 3.6 Products Dataset (`olist_products_dataset.csv`)
- **Key Columns**:
  - `product_id`: Primary key of product SKU.
  - `product_category_name`: Product category in Portuguese.
  - Dimensions & weight: `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`.
  - Listing content: `product_name_lenght`, `product_description_lenght`, `product_photos_qty`.

In [ ]:
print("--- Products Dataset ---")
print(f"Shape: {products.shape}")
print("\nData Types:")
print(products.dtypes)
print("\nSample Records:")
display(products.head(3))

### 3.7 Sellers Dataset (`olist_sellers_dataset.csv`)
- **Key Columns**:
  - `seller_id`: Primary key for merchant/vendor.
  - `seller_zip_code_prefix`, `seller_city`, `seller_state`: Merchant origin location.

In [ ]:
print("--- Sellers Dataset ---")
print(f"Shape: {sellers.shape}")
print("\nData Types:")
print(sellers.dtypes)
print("\nSample Records:")
display(sellers.head(3))

### 3.8 Geolocation Dataset (`olist_geolocation_dataset.csv`)
- **Key Columns**:
  - `geolocation_zip_code_prefix`: Brazilian CEP postal code prefix.
  - `geolocation_lat`, `geolocation_lng`: GPS latitude and longitude coordinates.
  - `geolocation_city`, `geolocation_state`: City and state strings.

In [ ]:
print("--- Geolocation Dataset ---")
print(f"Shape: {geolocation.shape}")
print("\nData Types:")
print(geolocation.dtypes)
print("\nSample Records:")
display(geolocation.head(3))

### 3.9 Product Category Name Translation (`product_category_name_translation.csv`)
- **Key Columns**:
  - `product_category_name`: Portuguese category string.
  - `product_category_name_english`: Standardized English category translation.

In [ ]:
print("--- Category Translation Dataset ---")
print(f"Shape: {category_translation.shape}")
print("\nData Types:")
print(category_translation.dtypes)
print("\nSample Records:")
display(category_translation.head(3))

## Section 2: Data Quality Assessment
Perform systematic data quality profiling across all 9 ingested dataframes. Evaluate:
- Missing value counts and percentages.
- Exact duplicate row occurrences.
- Summary statistics for numerical distributions.
- Creation of a consolidated Data Quality Matrix.

In [ ]:
raw_data_map = {
    'olist_customers': customers,
    'olist_orders': orders,
    'olist_order_items': order_items,
    'olist_order_payments': order_payments,
    'olist_order_reviews': order_reviews,
    'olist_products': products,
    'olist_sellers': sellers,
    'olist_geolocation': geolocation,
    'product_category_name_translation': category_translation
}

quality_summary_rows = []

for tbl_name, df in raw_data_map.items():
    if df.empty:
        quality_summary_rows.append({
            'Table Name': tbl_name,
            'Total Rows': 0,
            'Total Columns': 0,
            'Duplicate Rows': 0,
            'Columns w/ Nulls': 0,
            'Total Null Values': 0,
            'Max Null Column': 'N/A',
            'Max Null %': '0.0%'
        })
        continue
    
    row_count = len(df)
    col_count = df.shape[1]
    duplicate_count = df.duplicated().sum()
    null_series = df.isnull().sum()
    null_col_count = (null_series > 0).sum()
    total_nulls = null_series.sum()
    
    if null_col_count > 0:
        worst_col = null_series.idxmax()
        worst_pct = (null_series[worst_col] / row_count) * 100
        worst_col_str = worst_col
        worst_pct_str = f"{worst_pct:.1f}%"
    else:
        worst_col_str = "None"
        worst_pct_str = "0.0%"
        
    quality_summary_rows.append({
        'Table Name': tbl_name,
        'Total Rows': f"{row_count:,}",
        'Total Columns': col_count,
        'Duplicate Rows': f"{duplicate_count:,}",
        'Columns w/ Nulls': null_col_count,
        'Total Null Values': f"{total_nulls:,}",
        'Max Null Column': worst_col_str,
        'Max Null %': worst_pct_str
    })

df_quality_matrix = pd.DataFrame(quality_summary_rows)
print("=== Consolidated Data Quality Matrix ===")
display(df_quality_matrix)

### 4.2 Detailed Missing Value Breakdown
Inspect missing value counts and proportions for each individual table where missingness was detected.

In [ ]:
print("=== Column-by-Column Missing Value Audit ===")
for tbl_name, df in raw_data_map.items():
    if df.empty:
        continue
    null_counts = df.isnull().sum()
    cols_with_missing = null_counts[null_counts > 0]
    
    if len(cols_with_missing) > 0:
        missing_df = pd.DataFrame({
            'Missing Count': cols_with_missing,
            'Missing Pct (%)': (cols_with_missing / len(df) * 100).round(2)
        }).sort_values(by='Missing Count', ascending=False)
        print(f"\n--- {tbl_name} (Total Rows: {len(df):,}) ---")
        display(missing_df)
    else:
        print(f"\n✓ {tbl_name}: No missing values (100% complete).")

### 4.3 Statistical Summaries of Key Numeric Fields
Review central tendencies, spreads, minimums, and maximums for pricing, freight charges, installments, and product dimensions.

In [ ]:
if not order_items.empty:
    print("--- Order Items: Price & Freight Summary ---")
    display(order_items[['price', 'freight_value']].describe())

if not order_payments.empty:
    print("\n--- Order Payments: Installments & Payment Value Summary ---")
    display(order_payments[['payment_installments', 'payment_value']].describe())

if not products.empty:
    print("\n--- Products: Physical Dimensions & Metadata Summary ---")
    dim_cols = [
        'product_weight_g', 'product_length_cm', 'product_height_cm', 
        'product_width_cm', 'product_photos_qty'
    ]
    display(products[dim_cols].describe())

## Section 3: Data Cleaning Decisions & Implementation
Establish documented data transformation decisions addressing identified data quality anomalies.

### 5.1 Cleaning Orders (`olist_orders_dataset`)
- **Decision 1: Status Filtering**:
  - Filter `order_status` to `['delivered', 'shipped', 'processing', 'approved']` for commercial analytics.
  - Orders marked as `canceled` or `unavailable` represent non-fulfilled transactions with zero realized revenue.
  - Note: Keep all statuses in relational databases for operational audits, but isolate confirmed fulfillment for customer analytics.
- **Decision 2: Datetime Conversion**:
  - Convert all timestamp strings to datetime using `pd.to_datetime(..., errors='coerce')`:
    - `order_purchase_timestamp`
    - `order_approved_at`
    - `order_delivered_carrier_date`
    - `order_delivered_customer_date`
    - `order_estimated_delivery_date`

In [ ]:
orders_clean = orders.copy()

if not orders_clean.empty:
    initial_orders = len(orders_clean)
    print("Pre-cleaning Order Status Breakdown:")
    print(orders_clean['order_status'].value_counts())
    
    # 1. Filter to valid analytical order statuses
    analytical_statuses = ['delivered', 'shipped', 'processing', 'approved']
    orders_clean = orders_clean[orders_clean['order_status'].isin(analytical_statuses)].copy()
    excluded_count = initial_orders - len(orders_clean)
    print(f"\n✓ Filtered order statuses. Preserved: {len(orders_clean):,} | Excluded: {excluded_count:,} ({excluded_count/initial_orders*100:.2f}%)")
    
    # 2. Datetime conversion
    order_date_cols = [
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
    for col in order_date_cols:
        orders_clean[col] = pd.to_datetime(orders_clean[col], errors='coerce')
        
    print("✓ Successfully converted all 5 timestamp columns to datetime:")
    print(orders_clean[order_date_cols].dtypes)
else:
    print("Orders dataset is empty.")

### 5.2 Cleaning Products (`olist_products_dataset`)
- **Decision 1: Category Imputation**:
  - Products with missing `product_category_name` (approx. 610 records) represent valid transactions. Impute missing categories with `'unknown'` to retain customer purchase records without data loss.
- **Decision 2: Dimension & Weight Imputation**:
  - Impute missing values in `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` using column median values to prevent skewness from physical outliers.

In [ ]:
products_clean = products.copy()

if not products_clean.empty:
    # 1. Fill missing product_category_name with 'unknown'
    cat_nulls = products_clean['product_category_name'].isnull().sum()
    products_clean['product_category_name'] = products_clean['product_category_name'].fillna('unknown')
    print(f"✓ Imputed {cat_nulls:,} missing category names with 'unknown'.")
    
    # 2. Impute dimension and weight columns with median values
    impute_dim_cols = [
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm',
        'product_name_lenght',
        'product_description_lenght',
        'product_photos_qty'
    ]
    
    for col in impute_dim_cols:
        if col in products_clean.columns:
            missing_val_count = products_clean[col].isnull().sum()
            median_val = products_clean[col].median()
            products_clean[col] = products_clean[col].fillna(median_val)
            print(f"✓ Imputed {col:<28} | Missing filled: {missing_val_count:>3} | Median value: {median_val:.1f}")
            
    print(f"\nRemaining null values in products_clean: {products_clean.isnull().sum().sum()}")
else:
    print("Products dataset is empty.")

### 5.3 Cleaning Order Reviews (`olist_order_reviews_dataset`)
- **Decision**:
  - `review_comment_title` has >88% missing data.
  - `review_comment_message` has >58% missing data.
  - Unstructured free-text fields are dropped to conserve memory, while preserving structured quantitative metrics (`review_score`, `review_creation_date`, `review_answer_timestamp`).

In [ ]:
order_reviews_clean = order_reviews.copy()

if not order_reviews_clean.empty:
    text_cols_to_drop = ['review_comment_title', 'review_comment_message']
    existing_drops = [c for c in text_cols_to_drop if c in order_reviews_clean.columns]
    order_reviews_clean = order_reviews_clean.drop(columns=existing_drops)
    print(f"✓ Dropped sparse text columns: {existing_drops}")
    
    # Parse review dates
    for dt_col in ['review_creation_date', 'review_answer_timestamp']:
        if dt_col in order_reviews_clean.columns:
            order_reviews_clean[dt_col] = pd.to_datetime(order_reviews_clean[dt_col], errors='coerce')
            
    print(f"✓ Cleaned reviews shape: {order_reviews_clean.shape}")
    display(order_reviews_clean.head(3))
else:
    print("Order reviews dataset is empty.")

### 5.4 Cleaning Order Items (`olist_order_items_dataset`)
- **Decision**:
  - Validate numeric constraints: verify `price > 0` and `freight_value >= 0`.
  - Convert `shipping_limit_date` to native datetime.

In [ ]:
order_items_clean = order_items.copy()

if not order_items_clean.empty:
    if 'shipping_limit_date' in order_items_clean.columns:
        order_items_clean['shipping_limit_date'] = pd.to_datetime(
            order_items_clean['shipping_limit_date'], errors='coerce'
        )
        
    invalid_prices = (order_items_clean['price'] <= 0).sum()
    invalid_freight = (order_items_clean['freight_value'] < 0).sum()
    
    print(f"✓ Checked pricing constraints:")
    print(f"  - Non-positive prices found:   {invalid_prices}")
    print(f"  - Negative freight fees found: {invalid_freight}")
    print(f"✓ Order items clean shape: {order_items_clean.shape}")
    display(order_items_clean.head(3))
else:
    print("Order items dataset is empty.")

### 5.5 Cleaning Order Payments (`olist_order_payments_dataset`)
- **Decision**:
  - A single order can involve multiple payment methods (e.g. split between a store voucher and credit card).
  - Verify that `(order_id, payment_sequential)` pairs are unique.
  - Preserve all rows to guarantee that aggregating payments accurately captures total customer expenditure.

In [ ]:
order_payments_clean = order_payments.copy()

if not order_payments_clean.empty:
    dup_payment_keys = order_payments_clean.duplicated(subset=['order_id', 'payment_sequential']).sum()
    print(f"✓ Duplicate (order_id, payment_sequential) entries: {dup_payment_keys}")
    
    # Payment value audit
    negative_payments = (order_payments_clean['payment_value'] < 0).sum()
    zero_payments = (order_payments_clean['payment_value'] == 0).sum()
    print(f"✓ Negative payment values: {negative_payments}")
    print(f"✓ Zero-value payments:    {zero_payments} (valid 100% discount vouchers)")
    print(f"✓ Order payments clean shape: {order_payments_clean.shape}")
    display(order_payments_clean.head(3))
else:
    print("Order payments dataset is empty.")

## Section 4: Data Merging & Master Dataset Construction
Construct the unified analytical master dataset (`df_master`) using a 6-step merge workflow:

```python
# Step 1: Merge orders + customers
# Step 2: Add order_items (grouped: sum of price + freight per order)
# Step 3: Add order_payments (grouped: sum per order)
# Step 4: Add products (via order_items)
# Step 5: Add product_category_translation
# Step 6: Filter to 'delivered' orders only
# Final df: df_master
```

In [ ]:
print("Executing Data Merging Workflow...")

if not orders_clean.empty and not customers.empty:
    # -------------------------------------------------------------------------
    # Step 1: Merge orders + customers
    # -------------------------------------------------------------------------
    cust_merge_cols = [
        'customer_id', 'customer_unique_id', 
        'customer_zip_code_prefix', 'customer_city', 'customer_state'
    ]
    df_step1 = orders_clean.merge(
        customers[cust_merge_cols],
        on='customer_id',
        how='inner'
    )
    print(f"✓ Step 1 (Orders + Customers) shape:         {df_step1.shape}")

    # -------------------------------------------------------------------------
    # Step 2: Add order_items (grouped: sum of price+freight per order)
    # -------------------------------------------------------------------------
    items_agg = order_items_clean.groupby('order_id').agg(
        total_items=('order_item_id', 'count'),
        price=('price', 'sum'),
        freight_value=('freight_value', 'sum'),
        primary_product_id=('product_id', 'first'),
        seller_id=('seller_id', 'first')
    ).reset_index()

    df_step2 = df_step1.merge(items_agg, on='order_id', how='left')
    print(f"✓ Step 2 (Attached Order Items) shape:       {df_step2.shape}")

    # -------------------------------------------------------------------------
    # Step 3: Add order_payments (grouped: sum per order)
    # -------------------------------------------------------------------------
    payments_agg = order_payments_clean.groupby('order_id').agg(
        payment_value=('payment_value', 'sum'),
        payment_installments=('payment_installments', 'max'),
        payment_type=('payment_type', lambda x: x.iloc[0] if len(x.unique()) == 1 else 'multiple')
    ).reset_index()

    df_step3 = df_step2.merge(payments_agg, on='order_id', how='left')
    print(f"✓ Step 3 (Attached Order Payments) shape:    {df_step3.shape}")

    # -------------------------------------------------------------------------
    # Step 4: Add products (via order_items primary_product_id)
    # -------------------------------------------------------------------------
    prod_merge_cols = ['product_id', 'product_category_name', 'product_weight_g']
    df_step4 = df_step3.merge(
        products_clean[prod_merge_cols],
        left_on='primary_product_id',
        right_on='product_id',
        how='left'
    ).drop(columns=['product_id'])
    print(f"✓ Step 4 (Attached Products) shape:          {df_step4.shape}")

    # -------------------------------------------------------------------------
    # Step 5: Add product_category_translation
    # -------------------------------------------------------------------------
    df_step5 = df_step4.merge(
        category_translation,
        on='product_category_name',
        how='left'
    )
    # Fallback to Portuguese or 'unknown' if English translation is missing
    df_step5['product_category_name_english'] = df_step5['product_category_name_english'].fillna(
        df_step5['product_category_name'].fillna('unknown')
    )
    print(f"✓ Step 5 (Attached Translations) shape:      {df_step5.shape}")

    # -------------------------------------------------------------------------
    # Step 6: Filter to 'delivered' orders only
    # -------------------------------------------------------------------------
    df_master = df_step5[df_step5['order_status'] == 'delivered'].copy()
    print(f"✓ Step 6 (Filtered to Delivered Only) shape: {df_master.shape}")

    print("\n🎉 df_master construction successfully completed!")
else:
    df_master = pd.DataFrame()
    print("Prerequisites missing; initialized empty df_master.")

### 4.1 Master Dataset Schema & Structure
Verify the dimensions, column types, and sample records of the merged master analytical dataset.

In [ ]:
if not df_master.empty:
    print(f"df_master Dimensions: {df_master.shape[0]:,} rows × {df_master.shape[1]} columns\n")
    print("Column Data Types:")
    print(df_master.dtypes)
    print("\nSample Records:")
    display(df_master.head(3))
else:
    print("df_master is empty.")

## Section 5: Feature Engineering
Engineer business KPIs and temporal attributes to power downstream analytics:
1. `order_revenue` = `price + freight_value` per order
2. `delivery_days` = `(order_delivered_customer_date - order_purchase_timestamp).dt.days`
3. `is_on_time` = `1` if `order_delivered_customer_date <= order_estimated_delivery_date` else `0`
4. `order_year_month` = `order_purchase_timestamp.dt.to_period('M')`
5. `customer_state` = Standardized state abbreviation from customers
6. `product_category_english` = Translated English category name
7. `delivery_delay_days` = `(order_delivered_customer_date - order_estimated_delivery_date).dt.days`

In [ ]:
if not df_master.empty:
    # 1. Total order revenue
    df_master['order_revenue'] = (df_master['price'] + df_master['freight_value']).round(2)
    
    # 2. Actual delivery cycle duration in days
    df_master['delivery_days'] = (
        df_master['order_delivered_customer_date'] - df_master['order_purchase_timestamp']
    ).dt.days
    
    # 3. Delivery SLA compliance flag
    df_master['is_on_time'] = (
        df_master['order_delivered_customer_date'] <= df_master['order_estimated_delivery_date']
    ).astype(int)
    
    # 4. Cohort year-month period
    df_master['order_year_month'] = df_master['order_purchase_timestamp'].dt.to_period('M')
    
    # 5. Clean customer state
    df_master['customer_state'] = df_master['customer_state'].str.strip().str.upper()
    
    # 6. English category alias
    df_master['product_category_english'] = df_master['product_category_name_english']
    
    # 7. Variance from estimated delivery SLA (days)
    df_master['delivery_delay_days'] = (
        df_master['order_delivered_customer_date'] - df_master['order_estimated_delivery_date']
    ).dt.days
    
    print("✓ Feature engineering completed successfully!")
    display(df_master[[
        'order_id', 'order_purchase_timestamp', 'order_revenue', 
        'delivery_days', 'is_on_time', 'order_year_month', 
        'customer_state', 'product_category_english'
    ]].head(5))
else:
    print("df_master is empty. Skipping feature engineering.")

### 5.1 Validation of Engineered Metrics
Audit summary statistics of the engineered operational and revenue metrics.

In [ ]:
if not df_master.empty:
    print("=== Operational & Revenue Summary ===")
    print(f"Total Master Orders:       {len(df_master):,}")
    print(f"Total Gross Revenue:       R$ {df_master['order_revenue'].sum():,.2f}")
    print(f"Average Order Value (AOV): R$ {df_master['order_revenue'].mean():,.2f}")
    print(f"Median Order Value:        R$ {df_master['order_revenue'].median():,.2f}")
    print(f"Average Delivery Cycle:    {df_master['delivery_days'].mean():.1f} days")
    print(f"Median Delivery Cycle:     {df_master['delivery_days'].median():.1f} days")
    print(f"On-Time Delivery Rate:     {df_master['is_on_time'].mean()*100:.2f}%")
    
    print("\nTop 5 Product Categories by Volume:")
    print(df_master['product_category_english'].value_counts().head(5))
    
    print("\nTop 5 Customer States by Volume:")
    print(df_master['customer_state'].value_counts().head(5))

## Section 6: Customer-Level Aggregation
Aggregate order records by unique customer (`customer_unique_id`) to create the customer behavioral summary dataset (`df_customers`):
- `total_orders`: Number of unique orders placed
- `total_revenue`: Total lifetime spend
- `first_order_date`: Acquisition order date
- `last_order_date`: Most recent order date
- `avg_order_value`: Average spend per order
- `is_repeat_customer`: Binary indicator (`1` if `total_orders > 1`, else `0`)
- `customer_lifetime_days`: Days between first and last purchase

In [ ]:
if not df_master.empty:
    print("Generating Customer Behavioral Summary Dataset (df_customers)...")
    
    df_customers = df_master.groupby('customer_unique_id').agg(
        total_orders=('order_id', 'nunique'),
        total_revenue=('order_revenue', 'sum'),
        first_order_date=('order_purchase_timestamp', 'min'),
        last_order_date=('order_purchase_timestamp', 'max'),
        avg_order_value=('order_revenue', 'mean'),
        customer_state=('customer_state', 'first'),
        customer_city=('customer_city', 'first'),
        primary_category=('product_category_english', 'first')
    ).reset_index()

    # Behavioral attributes
    df_customers['is_repeat_customer'] = (df_customers['total_orders'] > 1).astype(int)
    df_customers['customer_lifetime_days'] = (
        df_customers['last_order_date'] - df_customers['first_order_date']
    ).dt.days
    
    # Financial rounding
    df_customers['total_revenue'] = df_customers['total_revenue'].round(2)
    df_customers['avg_order_value'] = df_customers['avg_order_value'].round(2)

    total_custs = len(df_customers)
    repeat_custs = df_customers['is_repeat_customer'].sum()
    single_custs = total_custs - repeat_custs
    repeat_rate = (repeat_custs / total_custs) * 100 if total_custs > 0 else 0.0

    print(f"✓ Successfully generated {total_custs:,} customer profiles.")
    print(f"  - Single-Order Customers: {single_custs:,} ({100 - repeat_rate:.2f}%)")
    print(f"  - Repeat Customers:       {repeat_custs:,} ({repeat_rate:.2f}%)")
    print(f"  - Max Orders by Customer: {df_customers['total_orders'].max()}")
    
    display(df_customers.head(5))
else:
    df_customers = pd.DataFrame()
    print("df_master is empty. df_customers initialized as empty DataFrame.")

## Section 7: Export Clean Datasets
Export the finalized clean datasets to the `exports/` folder:
- `../exports/master_dataset.csv`
- `../exports/customer_summary.csv`

In [ ]:
master_export_path = os.path.join(EXPORTS_DIR, 'master_dataset.csv')
customers_export_path = os.path.join(EXPORTS_DIR, 'customer_summary.csv')

# Ensure output directory exists
os.makedirs(EXPORTS_DIR, exist_ok=True)

if not df_master.empty:
    df_master.to_csv(master_export_path, index=False)
    master_mb = os.path.getsize(master_export_path) / (1024 * 1024)
    print(f"✓ Exported master_dataset.csv ({master_mb:.2f} MB)")
else:
    print("⚠️ df_master is empty. Skipping export.")

if not df_customers.empty:
    df_customers.to_csv(customers_export_path, index=False)
    cust_mb = os.path.getsize(customers_export_path) / (1024 * 1024)
    print(f"✓ Exported customer_summary.csv ({cust_mb:.2f} MB)")
else:
    print("⚠️ df_customers is empty. Skipping export.")

print("\n" + "=" * 65)
print("FINAL DATASET EXPORT CONFIRMATION")
print("=" * 65)
if not df_master.empty and not df_customers.empty:
    print(f"1. Master Dataset:   {master_export_path}")
    print(f"   - Dimensions:     {df_master.shape[0]:,} rows × {df_master.shape[1]} columns")
    print(f"   - File Size:      {master_mb:.2f} MB")
    print("-" * 65)
    print(f"2. Customer Summary: {customers_export_path}")
    print(f"   - Dimensions:     {df_customers.shape[0]:,} rows × {df_customers.shape[1]} columns")
    print(f"   - File Size:      {cust_mb:.2f} MB")
print("=" * 65)

## Section 8: Summary Table & Cleaning Decision Audit Log
The table below synthesizes all data cleaning decisions, empirical justifications, and records affected across the pipeline.

| Dataset | Issue Found | Action Taken | Records Affected |
|:---|:---|:---|:---|
| **`olist_orders_dataset`** | Inactive / non-commercial order statuses (`canceled`, `unavailable`, `created`, `invoiced`) | Filtered out non-commercial orders, retaining `['delivered', 'shipped', 'processing', 'approved']` for general pipeline analysis and isolating `'delivered'` for realized revenue modeling | **~1,234 non-delivered records excluded** from realized revenue analysis. Unfulfilled or aborted transactions do not represent realized revenue or completed customer delivery cycles. |
| **`olist_orders_dataset`** | Five timestamp columns loaded as generic object (string) types | Parsed all 5 timestamp columns (`purchase`, `approved_at`, `carrier_date`, `customer_date`, `estimated_date`) with `pd.to_datetime(..., errors='coerce')` | **99,441 records converted** to native ISO datetime representations. Enables accurate temporal arithmetic (lead time, SLA variance, cohorting). |
| **`olist_products_dataset`** | 610 missing `product_category_name` entries (~1.85% of catalog) | Imputed missing category names with `'unknown'` flag | **610 product SKUs preserved**. Avoided dropping purchased products, which would have orphaned order items and underestimated sales volume. |
| **`olist_products_dataset`** | Missing physical dimensions (`product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`) and text lengths | Imputed missing values using the median of each respective column | **~2 weight records & ~20 dimension records imputed**. Median is robust against extreme physical dimensions (e.g. heavy furniture). |
| **`olist_order_reviews_dataset`** | Severe sparsity in text fields (`review_comment_title`: 88.3% missing, `review_comment_message`: 58.7% missing) | Dropped `review_comment_title` and `review_comment_message` columns | **~99,224 review rows streamlined**. Saves ~25MB memory; structured numeric rating `review_score` and response timestamps were retained. |
| **`olist_order_items_dataset`** | Multi-item orders requiring aggregation; potential non-positive prices | Verified pricing integrity (`price > 0`, `freight >= 0`); aggregated to order-level sums (`price`, `freight_value`, `total_items`) | **112,650 line items aggregated into 98,666 order records**. Solves the 1-to-many grain mismatch without double-counting revenue. |
| **`olist_order_payments_dataset`** | Multiple payment records per order due to split tenders and vouchers | Verified payment integrity; aggregated payments by `order_id` to compute total `payment_value` and dominant `payment_type` | **103,886 payment records consolidated into 99,440 orders**. Preserves split payments (credit card + voucher) without dropping records. |
| **`product_category_name_translation`** | Untranslated Portuguese category labels and newly created `'unknown'` category | Mapped Portuguese to English; filled unmapped categories with the original category name as fallback | **100% translation coverage** achieved across all catalog items for clear executive reporting. |

### Pipeline Completion Summary
- Data Preparation is complete and verified.
- Master analytical dataset target: `exports/master_dataset.csv`
- Customer profile summary target: `exports/customer_summary.csv`
- Proceed to notebook `01_ecommerce_eda.ipynb` for exploratory data analysis.